In [ ]:
import uuid
import json
from pathlib import Path

import numpy as np
from skimage.transform import downscale_local_mean
import matplotlib.pyplot as plt
from pyboy import PyBoy
# from pyboy.logger import log_level
import mediapy as media
from einops import repeat

from gymnasium import Env, spaces
from pyboy.utils import WindowEvent

from global_map import local_to_global, GLOBAL_MAP_SHAPE
'''
red_gym_env 부분 공브
json은 파이썬의 딕셔너리, 리스트 같은 데이터 구조를 json 형식의 문자열로 변환함.
Encoding, Decoding

skimage.transform.downscale_local_mean: 이미지 처리 라이브러리인 scikit-image의 함수로, 이미지의 해상도를 낮추는 데 사용됩니다. AI 모델이 처리해야 할 데이터의 양을 줄여 학습 효율을 높입니다.
pyboy.PyBoy: 게임보이 에뮬레이터의 핵심 클래스입니다. 게임 롬을 불러오고, 실행하고, 게임 상태를 조작하는 모든 기능을 제공합니다.
mediapy as media: 동영상 파일을 쉽게 만들고 처리할 수 있게 해주는 라이브러리입니다. 에이전트의 플레이를 녹화하는 데 사용됩니다.
einops.repeat: 텐서(다차원 배열)의 형태를 유연하게 바꾸는 데 사용되는 라이브러리입니다. 여기서는 맵 이미지를 확대하는 데 사용됩니다.
gymnasium.Env, spaces: 강화학습 환경을 만들기 위한 표준 API입니다. Env는 환경 클래스의 기반이 되고, spaces는 행동 공간(action_space)과 관찰 공간(observation_space)을 정의하는 데 사용됩니다.
pyboy.utils.WindowEvent: 키보드 입력과 같은 이벤트를 에뮬레이터에 전달할 때 사용되는 상수들을 포함합니다
global_map...: global_map.py 파일에 정의된 함수와 변수를 가져옵니다. 게임 내의 지역 좌표(local)를 전체 월드맵의 전역 좌표(global)로 변환하는 역할을 합니다.
'''

In [ ]:
event_flags_start = 0xD747
event_flags_end = 0xD87E
museum_ticket = (0xD754, 0)
'''
ㅇㅎ..
event_flags_start: 게임 내 이벤트 플래그가 시작되는 메모리 주소입니다.
event_flags_end: 이벤트 플래그가 끝나는 메모리 주소입니다.
museum_ticket: 박물관 티켓 이벤트에 해당하는 특정 메모리 주소와 비트(bit) 위치입니다.
'''

In [ ]:
class RedGymEnv(Env):
    def __init__(self, config=None):
        self.s_path = config["session_path"]
        self.save_final_state = config["save_final_state"]
        self.print_rewards = config["print_rewards"]
        self.headless = config["headless"]
        self.init_state = config["init_state"]
        self.act_freq = config["action_freq"]
        self.max_steps = config["max_steps"]
        self.save_video = config["save_video"]
        self.fast_video = config["fast_video"]
        self.frame_stacks = 3
'''
config에서 받아오는 변수들
본격적인 RedGymEnv class 정의
init -> C++생성자랑 비슷함
config라는 딕셔너리에서 아까 봤던 값들을 할당함.
s_path : 저장될 파일의 경로
save_final_state : 최종 상태 저장 여부 -> True시 한 에피소드 끝날때 저장함(이미지 파일로..).
print_rewards : 보상 출력 여부 -> True 시 학습동안 현재 단게 보상값을 계속 출력 모니터링용.
headless : True시 에뮬레이터를 백그라운드에서만 실행
init_state : 초기 상태 파일 경로 reset 함수가 호출될 때마다 똑같은 지점에서 시작.
act_freq : 행동 빈도, 행동이 몇프레임 동안 유지 될지 정함. 단위: 프레임
max_steps : 에피소드 당 최대 스텝 수. 한 에피소드에서 AI가 행동할 수 있는 최대의 수
save_video : 동영상 저장 여부. True 시 AI의 플레이 과정을 저장함.
fast_video : 빠른 비디오 녹화 여부. 비디오저장할 때 성능에 영향 덜미침. 위에랑 아래랑 둘중 하나 쓰면 됨.
explore_weight: 탐험 보상 가중치 값을 높이면 새로운 장소를 방문하는 행동을 선호.
'''
        self.explore_weight = (
            1 if "explore_weight" not in config else config["explore_weight"]
        )
        self.reward_scale = (
            1 if "reward_scale" not in config else config["reward_scale"]
        )
        self.instance_id = (
            str(uuid.uuid4())[:8]
            if "instance_id" not in config
            else config["instance_id"]
        )
        self.s_path.mkdir(exist_ok=True)
        self.full_frame_writer = None
        self.model_frame_writer = None
        self.map_frame_writer = None
        self.reset_count = 0
        self.all_runs = []

        self.essential_map_locations = {
            v: i for i, v in enumerate([
                40, 0, 12, 1, 13, 51, 2, 54, 14, 59, 60, 61, 15, 3, 65
            ])
        }
'''
내부 변수
frame stack : 보상 스케일. 모든 보상 값에 공통적으로 곱해지는 전체 배율
instance_id : 병렬 시행시 각 환경을 구별하기위한 고유 ID
s_path.mkdir(exist_ok=True) : 폴더를 실제로 생성하는 코드. 폴더가 이미 존재해도 오류는 발생시키지 마라는 뜻..
full_frame_winter: 동영상 작성기 객체 처음에는 None이고 녹화가 시작되면 할당됨(객체들이).
reset_count : 말그대로 리셋 횟수 카운터 reset될 때마다 1씩 증가
all_runs : 모든 실행 기록,, 각 에피스드 끝날 때마다 그 결과 저장 통계, 분석을 위한것
essential_map_locations : 필수 맵 장소 게임 스토리 진행에 필수적인 맵들의 ID와 그 진행 순서를 딕셔너리로 저장한것. AI가 스토리를 순서대로 진행하면 보상을 줌.
'''
# Set this in SOME subclasses
        self.metadata = {"render.modes": []}
        self.reward_range = (0, 15000)

        self.valid_actions = [
            WindowEvent.PRESS_ARROW_DOWN,
            WindowEvent.PRESS_ARROW_LEFT,
            WindowEvent.PRESS_ARROW_RIGHT,
            WindowEvent.PRESS_ARROW_UP,
            WindowEvent.PRESS_BUTTON_A,
            WindowEvent.PRESS_BUTTON_B,
            WindowEvent.PRESS_BUTTON_START,
        ]

        self.release_actions = [
            WindowEvent.RELEASE_ARROW_DOWN,
            WindowEvent.RELEASE_ARROW_LEFT,
            WindowEvent.RELEASE_ARROW_RIGHT,
            WindowEvent.RELEASE_ARROW_UP,
            WindowEvent.RELEASE_BUTTON_A,
            WindowEvent.RELEASE_BUTTON_B,
            WindowEvent.RELEASE_BUTTON_START
        ]

        # load event names (parsed from https://github.com/pret/pokered/blob/91dc3c9f9c8fd529bb6e8307b58b96efa0bec67e/constants/event_constants.asm)
        with open("events.json") as f:
            event_names = json.load(f)
        self.event_names = event_names

        self.output_shape = (72, 80, self.frame_stacks)
        self.coords_pad = 12

        # Set these in ALL subclasses
        self.action_space = spaces.Discrete(len(self.valid_actions))

        self.enc_freqs = 8

        self.observation_space = spaces.Dict(
            {
                "screens": spaces.Box(low=0, high=255, shape=self.output_shape, dtype=np.uint8),
                "health": spaces.Box(low=0, high=1),
                "level": spaces.Box(low=-1, high=1, shape=(self.enc_freqs,)),
                "badges": spaces.MultiBinary(8),
                "events": spaces.MultiBinary((event_flags_end - event_flags_start) * 8),
                "map": spaces.Box(low=0, high=255, shape=(
                    self.coords_pad * 4, self.coords_pad * 4, 1), dtype=np.uint8),
                "recent_actions": spaces.MultiDiscrete([len(self.valid_actions)] * self.frame_stacks)
            }
        )
'''
metadata: 환경이 지원하는 렌더링 모드(사람이 볼 수 있는 창 모드인 'human')등을 명시하는 데 사용
reward_range: 한 스텝당 얻을 수 있는 보상의 이론적 최소값과 최대값 보상 스케일의 정구화
valid_actions: AI가 취할 수 있는 모든 행동 버튼 누르기의 리스트..
release_actions: valid_actions에 대응되는 버튼 떼기 행동의 리스트..
evnet.json 파일을 불러와 event_names 딕셔너리에 저장. 오박사에게서 포켓몬 받음 같은 문자열로 매핑해 준다.
output_shape: AI가 입력으로 받을 화면 이미지의 형태를 정의 (높이, 너비, 채널)
-> 원본 게임보이 화면 (144*160)을 절반으로 줄이고 3개의 프레임을 겹처서 사용한다.
coords_pad: 좌표 패딩 값 AI에게 보여줄 탑험 맵의 크기를 결정하는데 사용
플레이어의 현재 위치를 중심으로 상하좌우 12픽셀씩, 총 24x24 크기의 탐험 영역을 잘라내서 AI에게 보여주게 됩니다.
action_space: 행동 공간을 공식적으로 정의하는 부분, '강화학습 프레임워크에 필수적.
spaces.Discrete: AI의 행동이 이산적임을 의미. 여러 선택지 중 하나만 고르는 방식이다.
len(self.valid_actions)) -> 선택지를 valid actions 리스트 길이 (7개)로 지정.
enc_freqs: 인코딩 주파수..? 포켓몬 레벨 총합이라는 단일 숫자를 푸리에 인코딩할 때 사용할 주파수의 개수.값이 8이면 레벨 총합이라는 숫자 하나가 8개의 값을 가진 벡터로 변환되어 AI에게 전달.
observation_space: 관찰 공간을 공식적으로 정의하는 부분. AI가 매스텝마다 어떤 형태의 정보를 받을지 정의 근데 관찰 정보는
여기서는
"screens" 픽셀이고 (72,80,3) 형태의 3차원 배열. 값의 범위는 0~255
"health" 0과 1사이의 값을 가지는 숫자 하나
"level" enc_freqs 8개의 원소를 가지는 1차원배열..? 각 원소는 -1, 1사이의 값을 가짐
"badges" 8개의 0 또는 1로 이루어진 배열
"events" 모든 이벤트 플래그의 상태를 나타내는 0/1 배열
"map" (48, 48, 1) 형태의 2차원 배열 coords_pad를 기반으로 계산된 크기 AI에게 보여줄 주변 탐험 맵
"recent_actions" frame_stacks 3개 숫자로 이루어진 배열이고 AI가 직전에 취했던 3개 행동 기록
'''
        head = "null" if config["headless"] else "SDL2"

        # log_level("ERROR")
        self.pyboy = PyBoy(
            config["gb_path"],
            # debugging=False,
            # disable_input=False,
            window=head,
        )

        # self.screen = self.pyboy.botsupport_manager().screen()

        if not config["headless"]:
            self.pyboy.set_emulation_speed(6)
'''
head부분 "headless" 값에 따라 pyboy 모드 실행 결정
self.pyboy = PyBoy(
        config["gb_path"],
        window=head,
-> 실제 에뮬레이터를 실행하는 부분
# debugging=False -> 디버깅을 쓰지 않겠다
# disable_input = False -> 키 입력 허용 (AI나 사용자 입력 가능)
self.screen = self.pyboy.botsupport_manager().screen() -> 스크린 객체.. 스크린샷?

마지막줄은 게임창이 보인다면 에뮬레이션 속도를 6배로 설정 -> 이 설정은 학습속도자체에는 영향을 주지 않습니다?
'''

In [1]:
    def reset(self, seed=None, options={}):
        self.seed = seed
        # restart game, skipping credits
        with open(self.init_state, "rb") as f:
            self.pyboy.load_state(f)

        self.init_map_mem()

        self.agent_stats = []

        self.explore_map_dim = GLOBAL_MAP_SHAPE
        self.explore_map = np.zeros(self.explore_map_dim, dtype=np.uint8)

        self.recent_screens = np.zeros(self.output_shape, dtype=np.uint8)

        self.recent_actions = np.zeros((self.frame_stacks,), dtype=np.uint8)

        self.levels_satisfied = False
        self.base_explore = 0
        self.max_opponent_level = 0
        self.max_event_rew = 0
        self.max_level_rew = 0
        self.last_health = 1
        self.total_healing_rew = 0
        self.died_count = 0
        self.party_size = 0
        self.step_count = 0

        self.base_event_flags = sum([
            self.bit_count(self.read_m(i))
            for i in range(event_flags_start, event_flags_end)
        ])

        self.current_event_flags_set = {}

        # experiment!
        # self.max_steps += 128

        self.max_map_progress = 0
        self.progress_reward = self.get_game_state_reward()
        self.total_reward = sum([val for _, val in self.progress_reward.items()])
        self.reset_count += 1
        return self._get_obs(), {}
'''
말 그대로 리셋함수.
이 함수의 역할은 하나의 에피소드가 끝나고 다음 에피소드를 위해 깨끗하게 초기 상태로 돌리는 것
AI가 게임오버되거나, 주어진 시간을 다 썼을 때, "자, 이제 처음부터 다시 해보자!"라고 말하며 게임기와 주변 환경을 모두 정리하는 과정이라고 생각하시면 됩니다.
-> 에피소드는 한 판의 게임플레이
추가 옵션을 받을 수 있도록 설계됨.
init_map_mem() 맵과 관련된 메모리 변수를 초기화하는 다른 함수 호출
maybe -> seen_coords 딕셔너리를 비우는 작업 수행
self.agent_stats = []
이번 에피소드 동안 에이전트상태를 스텝마다 기록하는 리스트를 비움
self.explore_map_dim = GLOBAL_MAP_SHAPE
    self.explore_map = np.zeros(self.explore_map_dim, dtype=np.uint8)
AI가 방문한 위치를 기록하는 지도. np_zeros로 초기화
self.recent_screens = np.zeros(self.output_shape, dtype=np.uint8)
    self.recent_actions = np.zeros((self.frame_stacks,), dtype=np.uint8)
최근 행동들도 모두 0으로 초기화
self.levels_satisfied = False
    self.base_explore = 0
    self.max_opponent_level = 0
    self.max_event_rew = 0
    self.max_level_rew = 0
    self.last_health = 1
    self.total_healing_rew = 0
    self.died_count = 0
    self.party_size = 0
    self.step_count = 0
추적 변수도 모두 초기화
!!
self.base_event_flags = sum([
        self.bit_count(self.read_m(i))
        for i in range(event_flags_start, event_flags_end)
    ])
load_state로 초기 상태를 불러온 직후 이미 완료된 이벤트 플래그의 총 개수를 계산
base_event flags에 저장하여 기준점으로 삼는다
새롭게 달성한 이벤트에 대해서만 보상을 주어야 해서 만약 이미 완료되었으면 그건 보상계산에서 제외해야한다.

0번째 스텝에 대한 보상을 미리 계산해서
reset_count 1증가 시킴
return은 깨끗하게 초기화된 게임 상태에서 첫 번째 관찰을 생성..
리셋은 말 그대로 완전 새롭게 리셋되는데
AI가 똑똑해지려면 main에 있는 model 객체에서 똑똑해진다. 유일하게 전달되는 것은 reset_count뿐이다.
'''

'\n말 그대로 리셋함수.\n이 함수의 역할은 하나의 에피소드가 끝나고 다음 에피소드를 위해 깨끗하게 초기 상태로 돌리는 것\nAI가 게임오버되거나, 주어진 시간을 다 썼을 때, "자, 이제 처음부터 다시 해보자!"라고 말하며 게임기와 주변 환경을 모두 정리하는 과정이라고 생각하시면 됩니다.\n-> 에피소드는 한 판의 게임플레이\n추가 옵션을 받을 수 있도록 설계됨.\ninit_map_mem() 맵과 관련된 메모리 변수를 초기화하는 다른 함수 호출\nmaybe -> seen_coords 딕셔너리를 비우는 작업 수행\nself.agent_stats = []\n이번 에피소드 동안 에이전트상태를 스텝마다 기록하는 리스트를 비움\nself.explore_map_dim = GLOBAL_MAP_SHAPE\nself.explore_map = np.zeros(self.explore_map_dim, dtype=np.uint8)\nAI가 방문한 위치를 기록하는 지도. np_zeros로 초기화\nself.recent_screens = np.zeros(self.output_shape, dtype=np.uint8)\nself.recent_actions = np.zeros((self.frame_stacks,), dtype=np.uint8)\n'

In [ ]:
    def init_map_mem(self):
        self.seen_coords = {}
'''
seen_Coords : 본 적 있는 좌표들
이전에 탐험했던 것들을 모두 지우기 위해 있음. 새로운 에피소드를 위해서
'''

In [ ]:
    def render(self, reduce_res=True):
        # game_pixels_render = self.pyboy.screen.ndarray[:,:,0:1]  # (144, 160, 3)
        #game_pixels_render = self.pyboy.screen().screen_ndarray()[:, :, 0:1]
        game_pixels_render = self.pyboy.screen.ndarray[:, :, 0:1]

        if reduce_res:
            game_pixels_render = (
                downscale_local_mean(game_pixels_render, (2, 2, 1))
            ).astype(np.uint8)
        return game_pixels_render
'''
렌더링을 위해서 만들어진 함수인데
시각화를 담당하고 관찰데이터를 생성함.
reduce_res=True "해상도를 줄일지 결정" --> 굳이 필요가 없을 수도 있나?
game_pixels_render = self.pyboy.screen.ndarray[:, :, 0:1]
게임 화면을 Numpy 배열로 가져옴 게임보이 원본 해상도를 가져오게 됨. (144,160,1)
return game_pixels_render
'''

In [ ]:
    def _get_obs(self):

        screen = self.render()

        self.update_recent_screens(screen)

        # normalize to approx 0-1
        level_sum = 0.02 * sum([
            self.read_m(a) for a in [0xD18C, 0xD1B8, 0xD1E4, 0xD210, 0xD23C, 0xD268]
        ])

        observation = {
            "screens": self.recent_screens,
            "health": np.array([self.read_hp_fraction()]),
            "level": self.fourier_encode(level_sum),
            "badges": np.array([int(bit) for bit in f"{self.read_m(0xD356):08b}"], dtype=np.int8),
            "events": np.array(self.read_event_bits(), dtype=np.int8),
            "map": self.get_explore_map()[:, :, None],
            "recent_actions": self.recent_actions
        }

        return observation
'''
screen을 받고 render를 (numpy 배열이 받아지겠지) (72,80,1) 'reduce_res=True'
update_recent_screens(screen): screen에 최근 화면을 업데이트
3장의 프레임을 쌓아서 움직임 같은 동적인 정보를 파악함.
흠.. read_m은 만든거고 0xD18C, 0xD1B8이런거는 소지한 포켓몬스터 레벨 인 것 같음.
observation은 말그대로 객체를 하나 만드는 것 관찰 딕셔너리 관찰 공간인가 봄
'''

In [ ]:
    def step(self, action):

        if self.save_video and self.step_count == 0:
            self.start_video()

        self.run_action_on_emulator(action)
        self.append_agent_stats(action)

        self.update_recent_actions(action)

        self.update_seen_coords()

        self.update_explore_map()

        self.update_heal_reward()

        self.party_size = self.read_m(0xD163)

        new_reward = self.update_reward()

        self.last_health = self.read_hp_fraction()

        self.update_map_progress()

        step_limit_reached = self.check_if_done()

        obs = self._get_obs()
'''
강화학습 환경의 엔진 같은 부분이고 중요함.
AI가 할 행동 action을 변수로 받아옴.
비디오 녹화 조건.. 으로 녹화 할 지 정함.
run_action_on_emulator(action)로 입력받은 action을 실제로 게임에 적용함.
append_agent_status(action) 현재 스텝의 상세한 상태(위치,레벨,HP?)를 로그로 남김

그리고 recent_actions를 업데이트하고 seen_coord 방문한 좌표도 업데이트.
explore_map 탐험 지도도 업데이트 heal_rewards도 업데이트.
party_size = self.read_m(0xD163) -> 0xD163에 파티의 포켓몬 수가 저장되어 있나 봄.
new_reward -> 스텝하나 이동했으니까 reward도 변하고 update를 함.

self.last_health = self.read_hp_fraction()
-> 현재 HP 비율을 저장
self.update_map_progress() -> 스토리 진행도 업데이트..
step_limit_reached = self.check_if_done() -> 에피소드 최대 스탭에 도달했는지 확인
obs = self.get_obs() -> 한 스텝후 변화를 obs에 다시 넣음
'''

        # self.save_and_print_info(step_limit_reached, obs)
'''
여기 부분이 원래 print_rewards부분인데 지금 만든 친구가 빼놈
'''
        # create a map of all event flags set, with names where possible
        # if step_limit_reached:
        if self.step_count % 100 == 0:
            for address in range(event_flags_start, event_flags_end):
                val = self.read_m(address)
                for idx, bit in enumerate(f"{val:08b}"):
                    if bit == "1":
                        # TODO this currently seems to be broken!
                        key = f"0x{address:X}-{idx}"
                        if key in self.event_names.keys():
                            self.current_event_flags_set[key] = self.event_names[key]
                        else:
                            print(f"could not find key: {key}")

        self.step_count += 1
        return obs, new_reward, False, step_limit_reached, {}
'''
100스텝에 한번씩 실행함 무엇을?
val에서 1바이트 받아와서 for idx, bit에서 비트로 쪼개서 받음
enumerate(f"{val:08b}"): 8자리 이진수 문자열로 변환하는 거래 그후 enumerate가 그 idx를 받아옴.
만약 이벤트를 받아와서 1로 변경되었다면
# TODO ...: 개발자가 남긴 메모로, 이 기능이 현재 버그가 있거나 제대로 작동하지 않는 것 같다는 뜻입니다.

고유 키 생성: 이 이벤트 플래그를 고유하게 식별할 수 있는 문자열 키를 만듭니다.

{address:X}: 메모리 주소를 16진수 대문자(예: D747)로 변환합니다.

{idx}: 비트의 인덱스(0~7)를 붙입니다.

결과적으로 "0xD747-4" 와 같은 형태의 고유한 키가 만들어집니다.

이벤트 이름 찾기 및 저장:

if key in self.event_names.keys(): 위에서 만든 "0xD747-4"와 같은 키가 events.json 파일에서 불러온 self.event_names 딕셔너리에 존재하는지 확인합니다.

self.current_event_flags_set[key] = self.event_names[key]: 만약 키가 존재한다면, self.current_event_flags_set이라는 딕셔너리에 해당 이벤트의 정보를 추가합니다. 예를 들어 {"0xD747-4": "GOT_POKEDEX"} 와 같이 저장됩니다.

else: print(...): 만약 events.json에 해당 키에 대한 정보가 없다면, 키를 찾을 수 없다는 메시지를 출력합니다. 이는 아직 이름이 붙여지지 않은 이벤트를 발견했다는 의미이므로 디버깅에 유용합니다.

스탭 늘리고
return 함 뭐가 받겠지
obs, new_reward, False, step_limit_reached, {} -> False는 terminated 플래그라는데 뭐 몰라
{}는 info 딕셔너리이다.
'''
